# Questions 1.2.3 — Temps d'atteinte des premières limites indépendantes

**Cadre du modèle MODAL :**
- Deux files indépendantes : $Q_1$ côté ask et $Q_{-1}$ côté bid.
- Taux constants : $\lambda^+ = 1.2$ pour les ajouts et $\lambda^- = 1.5$ pour les retraits, par milliseconde.
- Condition initiale : $Q_1(0) = Q_{-1}(0) = 10$.
- On simule jusqu'au premier instant où l'une des deux files atteint 0.

L'objectif est d'étudier le temps d'atteinte
$$T_{\min} = \min(T_1, T_{-1}),$$
où $T_i = \inf\{t \ge 0 : Q_i(t)=0\}$.


## Fonctions utilisées depuis `model/hitting_times.py`

- `HittingTimeResult` : objet résultat d'une simulation. Il contient le temps d'atteinte, la file qui atteint 0 en premier, les trajectoires et les valeurs finales.
- `simulate_until_hit_zero` : simule deux files birth-death indépendantes jusqu'à ce que $Q_1$ ou $Q_{-1}$ atteigne 0.
- `simulate_brownian_until_hit_zero` : simule l'approximation brownienne $X(t)=x_0+\mu t+\sigma W(t)$ jusqu'à l'atteinte de 0.
- `hitting_time_pdf_brownian` : calcule la densité théorique du temps d'atteinte brownien, de type inverse gaussienne.
- `hitting_time_cdf_brownian` : calcule la fonction de répartition associée.
- `hitting_time_mean_brownian` et `hitting_time_variance_brownian` : donnent $\mathbb{E}[T_0]$ et $\mathrm{Var}(T_0)$ pour l'approximation brownienne.
- `prob_q1_hits_first_brownian` : approxime $\mathbb{P}(T_1<T_{-1})$ par intégration numérique.
- `batch_hitting_times` : répète plusieurs simulations et renvoie la moyenne empirique, l'écart-type, l'intervalle de confiance et la probabilité que $Q_1$ soit la première file vide.
- `scan_initial_conditions` : répète l'étude sur une grille de valeurs initiales $(Q_1(0), Q_{-1}(0))$.


In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np
import matplotlib.pyplot as plt
from model.hitting_times import *

plt.rcParams.update({
    'figure.figsize': (12, 4),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Paramètres du modèle birth-death.
LP = 1.2          # lambda+ : taux d'ajout, par ms
LM = 1.5          # lambda- : taux de retrait, par ms
MU = LP - LM      # drift de l'approximation brownienne
SIGMA2 = LP + LM  # taux de variance
SIGMA = np.sqrt(SIGMA2)
Q0 = 10           # taille initiale de chaque file

print(f'lambda+ = {LP}, lambda- = {LM}')
print(f'mu = lambda+ - lambda- = {MU}')
print(f'sigma^2 = lambda+ + lambda- = {SIGMA2}, sigma = {SIGMA:.4f}')
print(f'Q1(0) = Q-1(0) = {Q0}')


## Q0 — Simulation jusqu'à la première file vide

Chaque file évolue comme un processus birth-death :

$$Q_i(t)=Q_i(0)+N_i^+(t)-N_i^-(t).$$

Les deux files sont indépendantes. À chaque événement, une seule des quatre possibilités se produit : ajout dans $Q_1$, retrait dans $Q_1$, ajout dans $Q_{-1}$ ou retrait dans $Q_{-1}$.


In [ ]:
# Une simulation avec enregistrement complet des trajectoires.
rng = np.random.default_rng(42)
res = simulate_until_hit_zero(Q0, Q0, LP, LM, rng=rng, record_path=True)

which_label = 'Q_1' if res.which_hit == 1 else 'Q_{-1}'

print(f"Temps d'atteinte : T = {res.hitting_time:.2f} ms")
print(f'File atteignant 0 en premier : {which_label}')
print(f'Valeurs finales : Q1 = {res.q1_final}, Q-1 = {res.q_neg1_final}')
print(f"Nombre d'événements : {len(res.times) - 1}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.step(res.times, res.q1_path, where='post', lw=1, color='C3', label='$Q_1$ (ask)')
ax.step(res.times, res.q_neg1_path, where='post', lw=1, color='C0', label='$Q_{-1}$ (bid)')
ax.axhline(0, color='black', lw=0.5)
ax.scatter([res.hitting_time], [0], zorder=5, color='red', s=80, marker='x', linewidths=2)
ax.set_xlabel('Temps (ms)')
ax.set_ylabel('Taille de la file')
ax.set_title(f"Simulation jusqu'à la première file vide (T = {res.hitting_time:.2f} ms)")
ax.legend()
plt.tight_layout()
plt.show()


## Q1 — Approximation brownienne

Pour une file seule,

$$Q_i(t)=Q_i(0)+N_i^+(t)-N_i^-(t),$$

avec $N_i^+(t)\sim \mathrm{Poi}(\lambda^+t)$ et $N_i^-(t)\sim \mathrm{Poi}(\lambda^-t)$ indépendants.

On a donc :

$$\mathbb{E}[N_i^+(t)-N_i^-(t)] = (\lambda^+-\lambda^-)t = \mu t,$$

$$\mathrm{Var}(N_i^+(t)-N_i^-(t)) = (\lambda^+ + \lambda^-)t = \sigma^2t.$$

Par le théorème central limite fonctionnel,

$$\frac{N_i^+(t)-N_i^-(t)-\mu t}{\sigma\sqrt{t}} \Rightarrow \mathcal{N}(0,1),$$

et, au niveau des trajectoires,

$$Q_i(t) \approx Q_i(0)+\mu t+\sigma W(t),$$

où $W(t)$ est un mouvement brownien standard.


## Q2 — Distribution du temps d'atteinte : simulation vs théorie brownienne

Pour le mouvement brownien arithmétique

$$X(t)=x_0+\mu t+\sigma W(t), \qquad \mu<0,$$

le temps d'atteinte de 0,

$$T_0 = \inf\{t\ge0 : X(t)\le0\},$$

suit une loi inverse gaussienne. Sa densité est :

$$f_{T_0}(t)=\frac{x_0}{\sigma\sqrt{2\pi t^3}}\exp\left(-\frac{(x_0+\mu t)^2}{2\sigma^2t}\right).$$

Ses deux premiers moments sont :

$$\mathbb{E}[T_0]=\frac{x_0}{|\mu|}, \qquad \mathrm{Var}(T_0)=\frac{x_0\sigma^2}{|\mu|^3}.$$


In [ ]:
# On compare le temps d'atteinte d'une file birth-death et celui de son approximation brownienne.
N_sim = 5000
rng = np.random.default_rng(42)

# Temps d'atteinte pour une file birth-death seule.
ht_bd = []
for _ in range(N_sim):
    q = Q0
    t = 0.0
    while q > 0:
        dt = rng.exponential(1.0 / (LP + LM))
        t += dt
        q += 1 if rng.random() < LP / (LP + LM) else -1
    ht_bd.append(t)
ht_bd = np.array(ht_bd)

# Temps d'atteinte pour l'approximation brownienne.
ht_bm = []
dt_sim = 0.02
for _ in range(N_sim):
    t_hit, _, _ = simulate_brownian_until_hit_zero(Q0, MU, SIGMA, dt_sim, rng=rng)
    ht_bm.append(t_hit)
ht_bm = np.array(ht_bm)

# Densité théorique inverse gaussienne.
t_theorie = np.linspace(0.1, 150, 1000)
pdf_theorie = hitting_time_pdf_brownian(t_theorie, Q0, MU, SIGMA)

print(f'Birth-death : moyenne T = {ht_bd.mean():.2f} ms, écart-type = {ht_bd.std():.2f}')
print(f'Brownien simulé : moyenne T = {ht_bm.mean():.2f} ms, écart-type = {ht_bm.std():.2f}')
print(f'Espérance théorique : E[T] = {hitting_time_mean_brownian(Q0, MU):.2f}, '
      f'écart-type = {np.sqrt(hitting_time_variance_brownian(Q0, MU, SIGMA)):.2f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gauche : comparaison des densités.
ax = axes[0]
bins = np.linspace(0, 150, 60)
ax.hist(ht_bd, bins=bins, density=True, alpha=0.5, label='Birth-death simulé', color='C3')
ax.hist(ht_bm, bins=bins, density=True, alpha=0.5, label='Brownien simulé', color='C0')
ax.plot(t_theorie, pdf_theorie, 'k-', lw=2, label='Inverse gaussienne théorique')
ax.set_xlabel("Temps d'atteinte $T_0$ (ms)")
ax.set_ylabel('Densité')
ax.set_title("Distribution du temps d'atteinte d'une file")
ax.legend()
ax.set_xlim(0, 150)

# Droite : comparaison des fonctions de répartition.
ax = axes[1]
ht_bd_sorted = np.sort(ht_bd)
ht_bm_sorted = np.sort(ht_bm)
ecdf_bd = np.arange(1, N_sim + 1) / N_sim
ecdf_bm = np.arange(1, N_sim + 1) / N_sim
cdf_theorie = hitting_time_cdf_brownian(t_theorie, Q0, MU, SIGMA)

ax.step(ht_bd_sorted, ecdf_bd, lw=1, color='C3', label='Birth-death')
ax.step(ht_bm_sorted, ecdf_bm, lw=1, color='C0', label='Brownien simulé')
ax.plot(t_theorie, cdf_theorie, 'k--', lw=2, label='CDF théorique')
ax.set_xlabel('$T_0$ (ms)')
ax.set_ylabel('Fonction de répartition')
ax.set_title('Comparaison des fonctions de répartition')
ax.legend()
ax.set_xlim(0, 150)

plt.tight_layout()
plt.show()


## Q3 — Estimation de $\mathbb{E}[T_{\min}]$ et intervalle de confiance

On estime maintenant le temps moyen avant que l'une des deux files atteigne 0 :

$$T_{\min}=\min(T_1,T_{-1}).$$

La moyenne empirique est

$$\overline{T}_n = \frac{1}{n}\sum_{k=1}^n T_{\min}^{(k)}.$$

Par la loi des grands nombres,

$$\overline{T}_n \to \mathbb{E}[T_{\min}].$$

Pour l'intervalle de confiance, on utilise le TCL :

$$\sqrt{n}\,\frac{\overline{T}_n-\mathbb{E}[T_{\min}]}{s_n} \Rightarrow \mathcal{N}(0,1),$$

où $s_n$ est l'écart-type empirique. Par le lemme de Slutsky, remplacer l'écart-type théorique par $s_n$ donne l'intervalle asymptotique :

$$\overline{T}_n \pm 1.96\frac{s_n}{\sqrt{n}}.$$


In [ ]:
N_runs = 5000
batch = batch_hitting_times(N_runs, Q0, Q0, LP, LM, seed=42)

E_T_one = hitting_time_mean_brownian(Q0, MU)

print("=== Temps d'atteinte de min(Q1, Q-1) ===")
print(f'  Nombre de simulations : {N_runs}')
print(f'  Moyenne empirique :     {batch["mean_T"]:.2f} ms')
print(f'  Écart-type empirique :  {batch["std_T"]:.2f} ms')
print(f'  IC 95% :                [{batch["ci_95"][0]:.2f}, {batch["ci_95"][1]:.2f}] ms')
print(f'  P(Q1 en premier) :      {batch["prob_q1_first"]:.3f} (environ 0.5 par symétrie)')
print(f'\n  E[T] pour une seule file, théorie brownienne : {E_T_one:.2f} ms')
print('  Comme attendu, E[T_min] < E[T] pour une seule file.')


## Q4 — Influence de $Q_1(0)$ et $Q_{-1}(0)$

On répète l'estimation sur une grille de conditions initiales. L'idée est simple : si l'une des deux files commence très petite, le minimum $T_{\min}$ est généralement petit, même si l'autre file est grande.


In [ ]:
q_range = np.arange(2, 22, 2)
print(f'Grille : {len(q_range)} x {len(q_range)} = {len(q_range)**2} points')

scan = scan_initial_conditions(
    q_range,
    q_range,
    n_runs=500,
    lambda_plus=LP,
    lambda_minus=LM,
    seed=42,
)
print('Calcul terminé.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gauche : carte de chaleur de E[T_min].
ax = axes[0]
im = ax.imshow(
    scan['mean_T_grid'],
    origin='lower',
    aspect='auto',
    extent=[q_range[0]-1, q_range[-1]+1, q_range[0]-1, q_range[-1]+1],
    cmap='viridis',
)
plt.colorbar(im, ax=ax, label='$\\mathbb{E}[T_{\\min}]$ (ms)')
ax.set_xlabel('$Q_1(0)$')
ax.set_ylabel('$Q_{-1}(0)$')
ax.set_title('Temps moyen avant la premiere file vide')

# Droite : coupes a Q_{-1}(0) fixe.
ax = axes[1]
for q_neg1 in [4, 10, 20]:
    idx_j = np.argmin(np.abs(q_range - q_neg1))
    ax.plot(
        q_range,
        scan['mean_T_grid'][idx_j, :],
        'o-',
        ms=4,
        label=f'$Q_{{-1}}(0) = {q_range[idx_j]}$',
    )

# Ligne de reference : temps moyen theorique pour une seule file.
ax.plot(q_range, q_range / abs(MU), 'k--', lw=1.5,
        label='$x/|\\mu|$ (une seule file)')
ax.set_xlabel('$Q_1(0)$')
ax.set_ylabel('$\\mathbb{E}[T_{\\min}]$ (ms)')
ax.set_title('Coupes du temps moyen')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


## Q5 — Probabilité que $Q_1$ atteigne 0 avant $Q_{-1}$

Par symétrie, si $Q_1(0)=Q_{-1}(0)$, on s'attend à

$$\mathbb{P}(T_1<T_{-1})=0.5.$$

Si $Q_1(0)<Q_{-1}(0)$, la file ask commence plus petite, donc elle a plus de chances d'être la première vide.

Avec deux approximations browniennes indépendantes, on utilise :

$$\mathbb{P}(T_1<T_{-1}) = \int_0^\infty f_{T_1}(t)\,[1-F_{T_{-1}}(t)]\,dt.$$

Justification : $f_{T_1}(t)dt$ est la probabilité que $T_1$ tombe près de $t$, et $1-F_{T_{-1}}(t)$ est la probabilité que l'autre file n'ait pas encore atteint 0 à cet instant.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gauche : carte de chaleur de P(Q1 en premier).
ax = axes[0]
im = ax.imshow(
    scan['prob_q1_grid'],
    origin='lower',
    aspect='auto',
    extent=[q_range[0]-1, q_range[-1]+1, q_range[0]-1, q_range[-1]+1],
    cmap='RdBu_r',
    vmin=0,
    vmax=1,
)
plt.colorbar(im, ax=ax, label='$\\mathbb{P}(Q_1$ atteint 0 en premier$)$')
ax.set_xlabel('$Q_1(0)$')
ax.set_ylabel('$Q_{-1}(0)$')
ax.set_title('Probabilité que $Q_1$ soit la premiere file vide')

# Droite : coupes et comparaison avec la théorie brownienne.
ax = axes[1]
for j, q_neg1 in enumerate([4, 10, 20]):
    idx_j = np.argmin(np.abs(q_range - q_neg1))
    ax.plot(
        q_range,
        scan['prob_q1_grid'][idx_j, :],
        'o-',
        ms=4,
        label=f'Simulation : $Q_{{-1}}(0)={q_range[idx_j]}$',
    )
    p_theorie = [
        prob_q1_hits_first_brownian(float(x1), float(q_range[idx_j]), MU, SIGMA)
        for x1 in q_range
    ]
    ax.plot(q_range, p_theorie, '--', lw=1.5, color=f'C{j}', alpha=0.6)

ax.axhline(0.5, color='gray', ls=':', lw=1, label='50% (symétrie)')
ax.set_xlabel('$Q_1(0)$')
ax.set_ylabel('$\\mathbb{P}(Q_1$ en premier$)$')
ax.set_title('Simulation en trait plein, théorie brownienne en pointillé')
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()


## Synthèse

| Question | Résultat principal |
|----------|--------------------|
| Q0 | La simulation événementielle s'arrête bien quand $Q_1$ ou $Q_{-1}$ atteint 0. |
| Q1 | L'approximation brownienne utilise $\mu=\lambda^+-\lambda^-$ et $\sigma^2=\lambda^++\lambda^-$. |
| Q2 | Le temps d'atteinte brownien suit une loi inverse gaussienne. |
| Q3 | $\mathbb{E}[T_{\min}]$ est estimée par Monte Carlo avec un IC asymptotique à 95%. Elle est inférieure à $\mathbb{E}[T_0]$ pour une seule file. |
| Q4 | $\mathbb{E}[T_{\min}]$ augmente avec les deux tailles initiales, mais reste surtout dominée par la plus petite file. |
| Q5 | $\mathbb{P}(Q_1\text{ en premier})$ vaut environ 0.5 sur la diagonale et diminue lorsque $Q_1(0)$ devient plus grand que $Q_{-1}(0)$. |
